# Smart Document Q&A Assistant with RAG

This Google Colab notebook builds an end-to-end Retrieval-Augmented Generation workflow using Gemini API, LangChain, ChromaDB, SQLite, and Gradio.

## Features
- Upload PDF and TXT files
- Extract and chunk document text with `RecursiveCharacterTextSplitter`
- Generate embeddings with Gemini
- Store vectors in ChromaDB
- Retrieve relevant chunks with vector similarity
- Generate grounded answers with Gemini LLM
- Log queries and document metadata in SQLite
- Launch a polished Gradio frontend in Colab


## 1. Install Dependencies

Run this cell once in Google Colab to remove conflicting preinstalled packages and install a Colab-compatible dependency set for this project. After it finishes, restart the runtime once and then run the notebook from the top.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr

%pip -q uninstall -y \
  langchain \
  langgraph \
  langgraph-prebuilt \
  google-adk \
  google-generativeai \
  peft \
  sentence-transformers \
  transformers \
  tokenizers \
  huggingface-hub

%pip -q install -U \
  pandas==2.2.2 \
  pillow==10.4.0 \
  pytesseract==0.3.13 \
  pymupdf==1.24.9 \
  gradio==5.31.0 \
  chromadb==0.5.23 \
  opentelemetry-api==1.38.0 \
  opentelemetry-sdk==1.38.0 \
  opentelemetry-proto==1.38.0 \
  opentelemetry-exporter-otlp-proto-common==1.38.0 \
  langchain-text-splitters==0.3.8 \
  langchain-chroma==0.2.3 \
  google-genai==1.62.0 \
  pydantic==2.11.10

## 2. Imports and Configuration

This section imports the libraries, requests the Gemini API key securely, and prepares workspace paths.

In [ ]:
import os
import io
import json
import sqlite3
import hashlib
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Tuple, Dict, Any
from getpass import getpass

import fitz
import pandas as pd
import pytesseract
import gradio as gr
from google import genai
from PIL import Image

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

BASE_DIR = Path('/content/smart_doc_qa')
DATA_DIR = BASE_DIR / 'data'
UPLOAD_DIR = DATA_DIR / 'uploads'
SAMPLE_DIR = DATA_DIR / 'sample_docs'
CHROMA_DIR = BASE_DIR / 'chroma_store'
DB_PATH = BASE_DIR / 'rag_logs.sqlite'

for directory in [BASE_DIR, DATA_DIR, UPLOAD_DIR, SAMPLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass('Enter your Gemini API key: ')

EMBEDDING_MODEL_CANDIDATES = ['gemini-embedding-001', 'text-embedding-004']
LLM_MODEL_CANDIDATES = ['gemini-2.5-flash', 'gemini-2.0-flash', 'gemini-1.5-flash']

genai_client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])


class GeminiEmbeddings(Embeddings):
    def __init__(self, client: genai.Client, model: str):
        self.client = client
        self.model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        if not texts:
            return []
        result = self.client.models.embed_content(
            model=self.model,
            contents=texts
        )
        return [item.values for item in result.embeddings]

    def embed_query(self, text: str) -> List[float]:
        result = self.client.models.embed_content(
            model=self.model,
            contents=text
        )
        return result.embeddings[0].values


def build_embeddings_with_fallback() -> GeminiEmbeddings:
    last_error = None
    for model_name in EMBEDDING_MODEL_CANDIDATES:
        try:
            candidate = GeminiEmbeddings(genai_client, model_name)
            # Probe the model once so Colab fails early with a clear message if unsupported.
            candidate.embed_query('embedding connectivity check')
            print('Using embedding model:', model_name)
            return candidate
        except Exception as exc:
            last_error = exc
            print(f'Embedding model unavailable: {model_name} -> {exc}')
    raise RuntimeError(f'Unable to initialize a Gemini embedding model. Last error: {last_error}')


def build_llm_model_with_fallback() -> str:
    last_error = None
    for model_name in LLM_MODEL_CANDIDATES:
        try:
            response = genai_client.models.generate_content(
                model=model_name,
                contents='Reply with the single word: ready'
            )
            if getattr(response, 'text', '').strip():
                print('Using LLM model:', model_name)
                return model_name
        except Exception as exc:
            last_error = exc
            print(f'LLM model unavailable: {model_name} -> {exc}')
    raise RuntimeError(f'Unable to initialize a Gemini generation model. Last error: {last_error}')


embeddings = build_embeddings_with_fallback()
LLM_MODEL = build_llm_model_with_fallback()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=['\n\n', '\n', '. ', ' ', '']
)

print('Workspace ready:', BASE_DIR)
print('Embedding model candidates:', EMBEDDING_MODEL_CANDIDATES)
print('LLM model candidates:', LLM_MODEL_CANDIDATES)
print('Active LLM model:', LLM_MODEL)

## 3. Initialize SQLite and Generate Sample Test Documents

The notebook creates the required SQLite tables and a few sample documents so the full pipeline can be tested immediately.

In [ ]:
def get_connection() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def initialize_database() -> None:
    with get_connection() as conn:
        conn.execute(
            '''
            CREATE TABLE IF NOT EXISTS query_logs (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                query TEXT NOT NULL,
                answer TEXT NOT NULL,
                sources TEXT,
                retrieved_chunks TEXT,
                latency_ms REAL,
                created_at TEXT NOT NULL
            )
            '''
        )
        existing_columns = {
            row['name'] for row in conn.execute("PRAGMA table_info(query_logs)").fetchall()
        }
        if 'retrieved_chunks' not in existing_columns:
            conn.execute("ALTER TABLE query_logs ADD COLUMN retrieved_chunks TEXT")
        conn.execute(
            '''
            CREATE TABLE IF NOT EXISTS document_registry (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                filename TEXT NOT NULL,
                filetype TEXT NOT NULL,
                char_count INTEGER NOT NULL,
                chunk_count INTEGER NOT NULL,
                doc_hash TEXT NOT NULL UNIQUE,
                ingested_at TEXT NOT NULL
            )
            '''
        )
        conn.execute(
            '''
            CREATE TABLE IF NOT EXISTS chunk_registry (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                filename TEXT NOT NULL,
                doc_hash TEXT NOT NULL,
                chunk_id INTEGER NOT NULL,
                chunk_text TEXT NOT NULL,
                char_count INTEGER NOT NULL,
                ingested_at TEXT NOT NULL
            )
            '''
        )


def create_sample_documents() -> List[Path]:
    samples: Dict[str, str] = {
        'company_policy.txt': (
            'BCommune Hybrid Work Policy\n\n'
            'Employees may work remotely up to three days per week with manager approval. '
            'Core collaboration hours are 11:00 AM to 4:00 PM IST from Monday to Friday. '
            'All customer data must be stored only in approved cloud systems and never on personal devices. '
            'Travel reimbursement claims must be submitted within 30 days of the trip completion date.'
        ),
        'product_overview.txt': (
            'SmartDocs Platform Overview\n\n'
            'SmartDocs is an internal knowledge assistant for policy, product, and support teams. '
            'The platform combines semantic search, retrieval ranking, and grounded generation. '
            'Primary use cases include employee support, compliance lookup, onboarding, and document summarization. '
            'The current release target is focused on fast document ingestion and auditable query logs.'
        )
    }

    written_files: List[Path] = []
    for filename, content in samples.items():
        file_path = SAMPLE_DIR / filename
        file_path.write_text(content, encoding='utf-8')
        written_files.append(file_path)
    return written_files


initialize_database()
sample_files = create_sample_documents()
print('SQLite database initialized at:', DB_PATH)
print('Sample files created:')
for sample in sample_files:
    print('-', sample)

## 4. Build the RAG Pipeline

These helpers extract text, chunk documents, index embeddings in ChromaDB, retrieve relevant context, generate answers, and store logs.

In [ ]:
vector_store = None


def compute_file_hash(file_bytes: bytes) -> str:
    return hashlib.sha256(file_bytes).hexdigest()


def extract_text_with_ocr_from_pdf(file_path: Path) -> str:
    try:
        pdf = fitz.open(file_path)
        ocr_pages = []
        for page in pdf:
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            image = Image.open(io.BytesIO(pix.tobytes('png')))
            page_text = pytesseract.image_to_string(image)
            if page_text.strip():
                ocr_pages.append(page_text.strip())
        pdf.close()
        return '\n\n'.join(ocr_pages).strip()
    except Exception as exc:
        raise ValueError(f'Failed OCR extraction from {file_path.name}: {exc}')


def extract_text_from_pdf(file_path: Path) -> str:
    try:
        pdf = fitz.open(file_path)
        text = []
        for page in pdf:
            text.append(page.get_text())
        pdf.close()
        extracted_text = '\n'.join(text).strip()
        if extracted_text:
            return extracted_text
        ocr_text = extract_text_with_ocr_from_pdf(file_path)
        if ocr_text:
            return ocr_text
        return ''
    except Exception as exc:
        raise ValueError(f'Failed to extract PDF text from {file_path.name}: {exc}')


def extract_text_from_txt(file_path: Path) -> str:
    try:
        return file_path.read_text(encoding='utf-8', errors='ignore').strip()
    except Exception as exc:
        raise ValueError(f'Failed to extract TXT text from {file_path.name}: {exc}')


def extract_document_text(file_path: Path) -> str:
    suffix = file_path.suffix.lower()
    if suffix == '.pdf':
        return extract_text_from_pdf(file_path)
    if suffix == '.txt':
        return extract_text_from_txt(file_path)
    raise ValueError(f'Unsupported file type: {suffix}')


def register_document(filename: str, filetype: str, char_count: int, chunk_count: int, doc_hash: str) -> None:
    with get_connection() as conn:
        conn.execute(
            '''
            INSERT OR REPLACE INTO document_registry (
                id, filename, filetype, char_count, chunk_count, doc_hash, ingested_at
            )
            VALUES (
                COALESCE((SELECT id FROM document_registry WHERE doc_hash = ?), NULL),
                ?, ?, ?, ?, ?, ?
            )
            ''',
            (doc_hash, filename, filetype, char_count, chunk_count, doc_hash, datetime.now(timezone.utc).isoformat())
        )


def clear_document_registry() -> None:
    with get_connection() as conn:
        conn.execute('DELETE FROM document_registry')


def clear_chunk_registry() -> None:
    with get_connection() as conn:
        conn.execute('DELETE FROM chunk_registry')


def register_chunk(filename: str, doc_hash: str, chunk_id: int, chunk_text: str) -> None:
    with get_connection() as conn:
        conn.execute(
            '''
            INSERT INTO chunk_registry (filename, doc_hash, chunk_id, chunk_text, char_count, ingested_at)
            VALUES (?, ?, ?, ?, ?, ?)
            ''',
            (
                filename,
                doc_hash,
                chunk_id,
                chunk_text,
                len(chunk_text),
                datetime.now(timezone.utc).isoformat()
            )
        )


def log_query(query: str, answer: str, sources: List[str], retrieved_chunks: List[Dict[str, Any]], latency_ms: float) -> None:
    with get_connection() as conn:
        conn.execute(
            '''
            INSERT INTO query_logs (query, answer, sources, retrieved_chunks, latency_ms, created_at)
            VALUES (?, ?, ?, ?, ?, ?)
            ''',
            (
                query,
                answer,
                ' | '.join(sources),
                json.dumps(retrieved_chunks, ensure_ascii=False),
                round(latency_ms, 2),
                datetime.now(timezone.utc).isoformat()
            )
        )


def fetch_logs(limit: int = 100) -> pd.DataFrame:
    with get_connection() as conn:
        query = '''
            SELECT id, query, answer, sources, retrieved_chunks, latency_ms, created_at
            FROM query_logs
            ORDER BY id DESC
            LIMIT ?
        '''
        return pd.read_sql_query(query, conn, params=(limit,))


def fetch_registered_documents() -> pd.DataFrame:
    with get_connection() as conn:
        query = '''
            SELECT id, filename, filetype, char_count, chunk_count, doc_hash, ingested_at
            FROM document_registry
            ORDER BY id DESC
        '''
        return pd.read_sql_query(query, conn)


def fetch_chunk_registry(limit: int = 200) -> pd.DataFrame:
    with get_connection() as conn:
        query = '''
            SELECT id, filename, chunk_id, char_count, chunk_text, ingested_at
            FROM chunk_registry
            ORDER BY filename ASC, chunk_id ASC
            LIMIT ?
        '''
        return pd.read_sql_query(query, conn, params=(limit,))


def reset_vector_store() -> None:
    global vector_store
    if CHROMA_DIR.exists():
        shutil.rmtree(CHROMA_DIR)
    CHROMA_DIR.mkdir(parents=True, exist_ok=True)
    vector_store = Chroma(
        collection_name='smart-doc-qa',
        persist_directory=str(CHROMA_DIR),
        embedding_function=embeddings
    )


def prepare_documents(file_paths: List[Path]) -> List[Document]:
    all_chunks: List[Document] = []
    for file_path in file_paths:
        text = extract_document_text(file_path)
        if not text:
            raise ValueError(
                f'No text could be extracted from {file_path.name}. The document may be empty, image-only, password-protected, or too low quality for OCR.'
            )

        file_bytes = file_path.read_bytes()
        doc_hash = compute_file_hash(file_bytes)
        chunks = text_splitter.split_text(text)
        if not chunks:
            raise ValueError(f'Chunking produced no output for {file_path.name}.')

        register_document(
            filename=file_path.name,
            filetype=file_path.suffix.lower().replace('.', ''),
            char_count=len(text),
            chunk_count=len(chunks),
            doc_hash=doc_hash
        )

        for index, chunk in enumerate(chunks):
            register_chunk(file_path.name, doc_hash, index, chunk)
            all_chunks.append(
                Document(
                    page_content=chunk,
                    metadata={
                        'source': file_path.name,
                        'chunk_id': index,
                        'doc_hash': doc_hash
                    }
                )
            )
    return all_chunks


def ingest_documents(file_paths: List[Path]) -> Tuple[int, str]:
    global vector_store
    if not file_paths:
        raise ValueError('Please provide at least one PDF or TXT file.')

    clear_document_registry()
    clear_chunk_registry()
    reset_vector_store()
    chunks = prepare_documents(file_paths)
    vector_store.add_documents(chunks)
    filenames = ', '.join(path.name for path in file_paths)
    return len(chunks), f'Indexed {len(chunks)} chunks from {len(file_paths)} documents: {filenames}.'


def build_context(docs: List[Document]) -> Tuple[str, List[str], List[Dict[str, Any]]]:
    sources = []
    retrieved_chunks = []
    context_blocks = []
    for doc in docs:
        source_name = doc.metadata.get('source', 'unknown')
        chunk_id = doc.metadata.get('chunk_id', 'n/a')
        sources.append(f'{source_name}#chunk-{chunk_id}')
        retrieved_chunks.append({
            'source': source_name,
            'chunk_id': chunk_id,
            'content': doc.page_content
        })
        context_blocks.append(f"Source: {source_name} | Chunk: {chunk_id}\n{doc.page_content}")
    return '\n\n'.join(context_blocks), sources, retrieved_chunks


def answer_question(query: str, k: int = 4) -> Tuple[str, List[str], str]:
    global vector_store
    if vector_store is None:
        raise ValueError('No vector store found. Please ingest documents before asking questions.')
    if not query or not query.strip():
        raise ValueError('Please enter a non-empty question.')

    start_time = time.perf_counter()
    docs = vector_store.similarity_search(query, k=k)
    if not docs:
        raise ValueError('No relevant chunks were retrieved for the question.')

    context, sources, retrieved_chunks = build_context(docs)
    prompt = f'''You are a helpful document question answering assistant.
Answer the user using only the provided context.
If the answer is not present in the context, say that it is not available in the uploaded documents.

Context:
{context}

Question: {query}

Return a concise, professional answer.'''

    response = genai_client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt
    )
    answer_text = response.text if getattr(response, 'text', None) else str(response)
    latency_ms = (time.perf_counter() - start_time) * 1000
    log_query(query, answer_text, sources, retrieved_chunks, latency_ms)
    status = f'Retrieved {len(docs)} chunks in {latency_ms:.2f} ms.'
    return answer_text, sources, status


## 5. Optional Sample Indexing

Sample files are generated for testing, but they are not indexed automatically. This keeps the uploaded-document flow clean. Run the next cell only if you want to test the pipeline with sample files.

In [ ]:
print('Sample files are ready but not indexed by default.')
print('If you want a quick test corpus, run: chunk_count, bootstrap_message = ingest_documents(sample_files)')
print('Current active registry:')
display(fetch_registered_documents())

## 6. Quick End-to-End Test

Run this only after documents have been indexed, either from the optional sample corpus or from your uploaded files.

In [ ]:
if vector_store is None:
    print('No documents are indexed yet. Upload files and index them first, or index sample_files manually.')
else:
    test_answer, test_sources, test_status = answer_question('Give me a concise summary of the indexed documents.')
    print('Answer:', test_answer)
    print('Sources:', test_sources)
    print('Status:', test_status)
    display(fetch_logs(limit=5))

## 7. Gradio Frontend

The interface below supports file uploads, chat-based Q&A, query log inspection, a clear chat button, and status indicators. It is designed to run directly in Google Colab.

In [ ]:
def resolve_uploaded_file_path(file_obj: Any) -> Path:
    if isinstance(file_obj, str):
        return Path(file_obj)
    if hasattr(file_obj, 'name') and file_obj.name:
        return Path(file_obj.name)
    if isinstance(file_obj, dict):
        for key in ['name', 'path']:
            if file_obj.get(key):
                return Path(file_obj[key])
    raise ValueError(f'Unsupported uploaded file object: {type(file_obj)}')


def save_uploaded_files(files: List[Any]) -> List[Path]:
    saved_paths: List[Path] = []
    for file_obj in files or []:
        source_path = resolve_uploaded_file_path(file_obj)
        if not source_path.exists():
            raise FileNotFoundError(f'Uploaded file path not found: {source_path}')
        target_path = UPLOAD_DIR / source_path.name
        shutil.copy(source_path, target_path)
        saved_paths.append(target_path)
    return saved_paths


def ingest_from_gradio(files: List[Any]) -> Tuple[str, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    try:
        if not files:
            return 'Status: Please upload at least one PDF or TXT file.', fetch_registered_documents(), fetch_chunk_registry(limit=200), fetch_logs(limit=100)

        saved_paths = save_uploaded_files(files)
        chunk_total, message = ingest_documents(saved_paths)
        status = f'Status: {message} Active RAG corpus updated. You can ask questions about these uploaded documents now.'
        print(f'Indexed chunks: {chunk_total}')
        return status, fetch_registered_documents(), fetch_chunk_registry(limit=200), fetch_logs(limit=100)
    except Exception as exc:
        return f'Status: Error during ingestion - {exc}', fetch_registered_documents(), fetch_chunk_registry(limit=200), fetch_logs(limit=100)


def chat_with_docs(message: str, history: List[Tuple[str, str]]) -> Tuple[List[Tuple[str, str]], str, pd.DataFrame]:
    history = history or []
    try:
        answer_text, sources, status = answer_question(message)
        source_block = '\n'.join(f'- {source}' for source in sources)
        assistant_reply = f"{answer_text}\n\nSources:\n{source_block}"
        history.append((message, assistant_reply))
        return history, f'Status: {status}', fetch_logs(limit=100)
    except Exception as exc:
        history.append((message, f'Error: {exc}'))
        return history, f'Status: {exc}', fetch_logs(limit=100)


def clear_chat() -> Tuple[List[Tuple[str, str]], str]:
    return [], 'Status: Chat cleared. Indexed documents remain available.'


def refresh_logs() -> pd.DataFrame:
    return fetch_logs(limit=100)


def refresh_registry() -> pd.DataFrame:
    return fetch_registered_documents()


def refresh_chunks() -> pd.DataFrame:
    return fetch_chunk_registry(limit=200)


custom_css = '''
.gradio-container {
    max-width: 1200px !important;
}
#status_box textarea {
    color: #0f172a !important;
    font-weight: 600;
}
'''


with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title='Smart Document Q&A Assistant') as demo:
    gr.Markdown(
        '''
        # Smart Document Q&A Assistant
        Upload PDF or TXT files, index them into ChromaDB with Gemini embeddings, and ask grounded questions.
        '''
    )

    chat_state = gr.State([])

    with gr.Tab('Assistant'):
        with gr.Row():
            with gr.Column(scale=1):
                file_input = gr.File(
                    label='Upload Documents',
                    file_count='multiple',
                    file_types=['.pdf', '.txt']
                )
                ingest_button = gr.Button('Index Documents', variant='primary')
                clear_button = gr.Button('Clear Chat')
                status_box = gr.Textbox(
                    label='System Status',
                    value='Status: No active documents indexed yet. Upload PDF or TXT files, then click Index Documents.',
                    interactive=False,
                    elem_id='status_box'
                )
                registry_df = gr.Dataframe(
                    value=fetch_registered_documents(),
                    label='Document Registry',
                    interactive=False,
                    wrap=True
                )
                chunk_df = gr.Dataframe(
                    value=fetch_chunk_registry(limit=200),
                    label='Indexed Chunks Preview',
                    interactive=False,
                    wrap=True
                )
                refresh_registry_button = gr.Button('Refresh Registry')
                refresh_chunks_button = gr.Button('Refresh Chunks')

            with gr.Column(scale=2):
                chatbot = gr.Chatbot(label='Document Chat', height=500)
                question_box = gr.Textbox(
                    label='Ask a question',
                    placeholder='Ask something about the uploaded documents...'
                )
                ask_button = gr.Button('Ask', variant='primary')

    with gr.Tab('Query Logs'):
        logs_df = gr.Dataframe(
            value=fetch_logs(limit=100),
            label='Recent Query Logs',
            interactive=False,
            wrap=True
        )
        refresh_logs_button = gr.Button('Refresh Logs')

    file_input.upload(
        fn=ingest_from_gradio,
        inputs=[file_input],
        outputs=[status_box, registry_df, chunk_df, logs_df]
    )

    ingest_button.click(
        fn=ingest_from_gradio,
        inputs=[file_input],
        outputs=[status_box, registry_df, chunk_df, logs_df]
    )

    ask_button.click(
        fn=chat_with_docs,
        inputs=[question_box, chat_state],
        outputs=[chatbot, status_box, logs_df]
    ).then(
        fn=lambda history: history,
        inputs=[chatbot],
        outputs=[chat_state]
    ).then(
        fn=lambda: '',
        inputs=None,
        outputs=[question_box]
    )

    question_box.submit(
        fn=chat_with_docs,
        inputs=[question_box, chat_state],
        outputs=[chatbot, status_box, logs_df]
    ).then(
        fn=lambda history: history,
        inputs=[chatbot],
        outputs=[chat_state]
    ).then(
        fn=lambda: '',
        inputs=None,
        outputs=[question_box]
    )

    clear_button.click(
        fn=clear_chat,
        inputs=None,
        outputs=[chatbot, status_box]
    ).then(
        fn=lambda: [],
        inputs=None,
        outputs=[chat_state]
    )

    refresh_logs_button.click(fn=refresh_logs, inputs=None, outputs=[logs_df])
    refresh_registry_button.click(fn=refresh_registry, inputs=None, outputs=[registry_df])
    refresh_chunks_button.click(fn=refresh_chunks, inputs=None, outputs=[chunk_df])

demo.launch(share=True)

## 8. Notes

- Uploaded documents become the active RAG corpus after you click `Index Documents`.
- Each new ingestion rebuilds the ChromaDB index and refreshes the active document registry.
- The `Indexed Chunks Preview` table shows the exact chunk records saved for the active uploaded corpus.
- Query logs store the question, answer, source chunks, retrieved chunk content, latency, and timestamp in SQLite at `/content/smart_doc_qa/rag_logs.sqlite`.
